In [1]:
import os
import torch
import json
from eval_utils_new import compute_classification_metrics, expected_calibration_error, approx_mrr_hits, mc_dropout_predict
from data_utils import load_vocab, load_triples_csv, build_edge_list_from_df, compute_2hop_drug_gene_disease_support
from models_new import ProposedRGCNModel, DistMult, ComplEx, SimpleGraphSAGE


c:\Users\Manasa\OneDrive\Desktop\Drug_Repurposing_Gnn\Drug_Repurposing_Gnn\.venv\lib\site-packages\torch_geometric\typing.py:86: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: [WinError 127] The specified procedure could not be found
  warnings.warn(f"An issue occurred while importing 'torch-scatter'. "
c:\Users\Manasa\OneDrive\Desktop\Drug_Repurposing_Gnn\Drug_Repurposing_Gnn\.venv\lib\site-packages\torch_geometric\typing.py:97: UserWarning: An issue occurred while importing 'torch-cluster'. Disabling its usage. Stacktrace: [WinError 127] The specified procedure could not be found
  warnings.warn(f"An issue occurred while importing 'torch-cluster'. "
c:\Users\Manasa\OneDrive\Desktop\Drug_Repurposing_Gnn\Drug_Repurposing_Gnn\.venv\lib\site-packages\torch_geometric\typing.py:113: UserWarning: An issue occurred while importing 'torch-spline-conv'. Disabling its usage. Stacktrace: [WinError 127] The specified procedure could not be found
 

In [ ]:

# 1. Import Libraries
import os, json
import numpy as np
import torch
from sklearn.metrics import roc_auc_score, average_precision_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# 2. Metrics

def sigmoid(x): 
    return 1.0 / (1.0 + np.exp(-x))

def compute_classification_metrics(y_true, y_prob):
    """AUROC, AUPRC"""
    try:
        auroc = roc_auc_score(y_true, y_prob)
    except Exception:
        auroc = float('nan')
    try:
        auprc = average_precision_score(y_true, y_prob)
    except Exception:
        auprc = float('nan')
    return auroc, auprc

def expected_calibration_error(y_true, y_prob, n_bins=10):
    """ECE with n_bins equal-width bins"""
    y_true = np.asarray(y_true); y_prob = np.asarray(y_prob)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        idx = np.where((y_prob >= bins[i]) & (y_prob < bins[i + 1]))[0]
        if len(idx) == 0: 
            continue
        acc = y_true[idx].mean()
        conf = y_prob[idx].mean()
        ece += (len(idx) / len(y_true)) * abs(acc - conf)
    return float(ece)


# Approximate Ranking (MRR, Hits@10)

def approx_mrr_hits(model, pos_triples, num_entities, edge_index=None, edge_type=None,
                    negatives_per_pos=200, device="cpu"):
    model.eval()
    mrrs, hits = [], 0
    with torch.no_grad():
        for (h, r, t) in pos_triples:
            h_t = torch.tensor([h], dtype=torch.long).to(device)
            r_t = torch.tensor([r], dtype=torch.long).to(device)
            t_pos = torch.tensor([t], dtype=torch.long).to(device)

            # sample negatives
            negs = np.random.choice(num_entities, size=negatives_per_pos, replace=False)
            negs_t = torch.tensor(negs, dtype=torch.long).to(device)

            # batch: pos + negs
            heads = torch.cat([h_t, h_t.repeat(negs_t.shape[0])], dim=0)
            rels  = torch.cat([r_t, r_t.repeat(negs_t.shape[0])], dim=0)
            tails = torch.cat([t_pos, negs_t], dim=0)

            try:
                logits = model(heads, rels, tails)  # baseline models
            except TypeError:
                logits = model((heads, rels, tails), edge_index.to(device), edge_type.to(device))  # GNN models

            if isinstance(logits, tuple):
                logits = logits[0]

            scores = logits.cpu().numpy()
            pos_score = scores[0]
            neg_scores = scores[1:]

            rank = 1 + int((neg_scores > pos_score).sum())
            mrrs.append(1.0 / rank)
            if rank <= 10:
                hits += 1

    return float(np.mean(mrrs)), float(hits / len(pos_triples)) if len(pos_triples) > 0 else (0.0, 0.0)



# Validation/Test Wrapper

def evaluate(model, val_triples, test_triples, edge_index, edge_type, device,
             num_entities=None, negatives_per_pos=100):
    metrics = {}

    # === Validation with neg sampling
    y_true, y_prob = [], []
    model.eval()
    with torch.no_grad():
        for (h, r, t) in val_triples:
            h_t = torch.tensor([h], dtype=torch.long).to(device)
            r_t = torch.tensor([r], dtype=torch.long).to(device)
            t_pos = torch.tensor([t], dtype=torch.long).to(device)

            # positive
            try:
                pos_logit = model(h_t, r_t, t_pos)
            except TypeError:
                pos_logit = model((h_t, r_t, t_pos), edge_index.to(device), edge_type.to(device))
            if isinstance(pos_logit, tuple): pos_logit = pos_logit[0]
            y_true.append(1)
            y_prob.append(torch.sigmoid(pos_logit).item())

            # negatives
            negs = np.random.choice(num_entities, size=negatives_per_pos, replace=False)
            negs_t = torch.tensor(negs, dtype=torch.long).to(device)
            h_batch = h_t.repeat(len(negs_t))
            r_batch = r_t.repeat(len(negs_t))
            try:
                neg_logits = model(h_batch, r_batch, negs_t)
            except TypeError:
                neg_logits = model((h_batch, r_batch, negs_t), edge_index.to(device), edge_type.to(device))
            if isinstance(neg_logits, tuple): neg_logits = neg_logits[0]
            y_true.extend([0] * len(negs))
            y_prob.extend(torch.sigmoid(neg_logits).cpu().numpy().tolist())

    auroc, auprc = compute_classification_metrics(y_true, y_prob)
    ece = expected_calibration_error(y_true, y_prob)
    metrics["val"] = {"AUROC": auroc, "AUPRC": auprc, "ECE": ece}

    # Test ranking
    mrr, hits10 = approx_mrr_hits(model, test_triples, num_entities,
                                  edge_index=edge_index, edge_type=edge_type,
                                  negatives_per_pos=negatives_per_pos, device=device)
    metrics["test"] = {"MRR": mrr, "Hits@10": hits10}

    return metrics


# Path Alignment
def path_alignment_score(model, triples, pair_support, edge_index, edge_type, device, ent2id=None):
    """
    Compute path alignment score.
    
    Args:
        model: trained KG model
        triples: list of (h, r, t) with entity IDs
        pair_support: dict with keys as (h,t) pairs -> score
                      (can be entity names or IDs)
        edge_index, edge_type: graph structure (if required by GNNs)
        device: torch device
        ent2id: optional mapping {entity_name: id} to handle raw string pairs
    """
    if len(triples) == 0:
        return 0.0

    # Map pair_support keys to IDs if needed
    mapped_support = {}
    for (h, t), score in pair_support.items():
        if isinstance(h, str) and ent2id is not None:
            if h not in ent2id or t not in ent2id:
                continue
            h, t = ent2id[h], ent2id[t]
        mapped_support[(int(h), int(t))] = float(score)

    heads, rels, tails = zip(*triples)
    heads = torch.tensor(heads).to(device)
    rels = torch.tensor(rels).to(device)
    tails = torch.tensor(tails).to(device)

    with torch.no_grad():
        try:
            preds = torch.sigmoid(model(heads, rels, tails))
        except TypeError:
            preds = torch.sigmoid(model((heads, rels, tails), edge_index.to(device), edge_type.to(device)))

    # Collect support scores for each triple
    support = [mapped_support.get((int(h), int(t)), 0.0)
               for h, t in zip(heads.cpu().numpy(), tails.cpu().numpy())]
    support = torch.tensor(support).float().to(device)

    matched = int((support > 0).sum().item())
    print(f"Path Alignment Debug: {matched}/{len(triples)} triples matched pair_support")

    if matched == 0:
        return 0.0

    return float((preds * support).mean().item())


# Load model helper
def load_model(ModelClass, model_path, num_entities, num_relations, dim, device):
    model = ModelClass(num_entities, num_relations, dim=dim, dropout=0.3).to(device)

    checkpoint = torch.load(model_path, map_location=device, weights_only=False)
    if isinstance(checkpoint, dict):
        if "model_state_dict" in checkpoint:
            state_dict = checkpoint["model_state_dict"]
        elif "model_state" in checkpoint:
            state_dict = checkpoint["model_state"]
        else:
            state_dict = checkpoint
    else:
        state_dict = checkpoint

    model.load_state_dict(state_dict, strict=True)
    model.eval()
    return model


Using device: cuda


In [3]:
# 3. Load Data


data_dir = r"C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/processed_graph"
data_dir2 = r"C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data"

# vocab
ent2id, id2ent, rel2id, id2rel = load_vocab(
    os.path.join(data_dir, "entities.txt"),
    os.path.join(data_dir, "relations.txt")
)

# triples
train_df = load_triples_csv(os.path.join(data_dir2, "train_inductive.csv"))
val_df   = load_triples_csv(os.path.join(data_dir2, "val_inductive.csv"))
test_df  = load_triples_csv(os.path.join(data_dir2, "test_inductive.csv"))

val_triples  = [(ent2id[h], rel2id[r], ent2id[t]) for h, r, t in val_df.values if h in ent2id and t in ent2id and r in rel2id]
test_triples = [(ent2id[h], rel2id[r], ent2id[t]) for h, r, t in test_df.values if h in ent2id and t in ent2id and r in rel2id]

edge_index, edge_type = build_edge_list_from_df(train_df, ent2id, rel2id)
_, pair_support = compute_2hop_drug_gene_disease_support(train_df, ent2id)

print(f"#Entities: {len(ent2id)} | #Relations: {len(rel2id)}")
print(f"#Validation triples: {len(val_triples)} | #Test triples: {len(test_triples)}")


#Entities: 94046 | #Relations: 107
#Validation triples: 582709 | #Test triples: 4855


In [4]:

# 4. Define Model Configs


model_configs = {
    "distmult": (DistMult, "C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/notebooks/checkpoints/distmult.pt"),
    "complex": (ComplEx, "C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/notebooks/checkpoints/complex.pt"),
    "graphsage": (SimpleGraphSAGE, "C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/notebooks/checkpoints/graphsage.pt"),
    "proposed_rgcn": (ProposedRGCNModel, "C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/notebooks/checkpoints/proposed_rgcn.pt"),
}


In [5]:

# 5. Run Evaluation

all_metrics = {}

for name, (ModelClass, path) in model_configs.items():
    if not os.path.exists(path):
        print(f"Skipping {name} — checkpoint not found: {path}")
        continue

    print(f"\n=== Evaluating {name.upper()} ===")
    model = load_model(ModelClass, path, len(ent2id), len(rel2id), dim=64, device=device)

    metrics = evaluate(model, val_triples, test_triples, edge_index, edge_type, device,
                       num_entities=len(ent2id), negatives_per_pos=100)

    alignment_val  = path_alignment_score(model, val_triples, pair_support, edge_index, edge_type, device)
    alignment_test = path_alignment_score(model, test_triples, pair_support, edge_index, edge_type, device)

    print("Metrics:", metrics)
    print(f"Path Alignment (val): {alignment_val:.3f}")
    print(f"Path Alignment (test): {alignment_test:.3f}")

    all_metrics[name] = {**metrics,
                         "path_alignment_val": alignment_val,
                         "path_alignment_test": alignment_test}

# save all results
os.makedirs("results", exist_ok=True)
with open("results/all_model_metrics.json", "w") as f:
    json.dump(all_metrics, f, indent=2)

print("\n Saved all metrics to results/all_model_metrics.json")



=== Evaluating DISTMULT ===
Path Alignment Debug: 0/582709 triples matched pair_support
Path Alignment Debug: 0/4855 triples matched pair_support
Metrics: {'val': {'AUROC': 0.9774887130057868, 'AUPRC': 0.30677818141840807, 'ECE': 0.06072076420879675}, 'test': {'MRR': 0.5841068787586644, 'Hits@10': 0.9775489186405767}}
Path Alignment (val): 0.000
Path Alignment (test): 0.000

=== Evaluating COMPLEX ===
Path Alignment Debug: 0/582709 triples matched pair_support
Path Alignment Debug: 0/4855 triples matched pair_support
Metrics: {'val': {'AUROC': 0.980741531831262, 'AUPRC': 0.36242740628170483, 'ECE': 0.058457219157096924}, 'test': {'MRR': 0.7093015697753616, 'Hits@10': 0.9604531410916581}}
Path Alignment (val): 0.000
Path Alignment (test): 0.000

=== Evaluating GRAPHSAGE ===
Path Alignment Debug: 0/582709 triples matched pair_support
Path Alignment Debug: 0/4855 triples matched pair_support
Metrics: {'val': {'AUROC': 0.9776089489237074, 'AUPRC': 0.27101604710545824, 'ECE': 0.08845268969

TypeError: ProposedRGCNModel.__init__() got an unexpected keyword argument 'dim'